# DINO (2021) 단독 실험

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

## 🎯 실험 목적

DINO의 특별한 특성 분석:
- ✅ Clean attention maps (artifacts 없음)
- ✅ 의미있는 segmentation
- ✅ 높은 feature 품질

## 📚 논문

**Emerging Properties in Self-Supervised Vision Transformers** (ICCV 2021)
- 저자: Mathilde Caron et al.
- arXiv: https://arxiv.org/abs/2104.14294

---

## 🚀 시작하기

### GPU 설정 (권장)
```
메뉴: 런타임 → 런타임 유형 변경 → GPU
```

### 실행 방법
- 메뉴: 런타임 → 모두 실행
- 또는 각 셀을 Shift + Enter로 순서대로 실행

## Step 1: 환경 설정 🔧

In [ ]:
# GPU 확인
!nvidia-smi

In [ ]:
# 라이브러리 설치
print("📦 라이브러리 설치 중...")
!pip install -q torch torchvision
print("✅ 완료!")

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Colab용
from google.colab import files
import urllib.request
import os

# 디바이스 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("\n" + "="*60)
print(f"🖥️  디바이스: {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  CPU 모드 (느릴 수 있음)")
print("="*60 + "\n")

# 시각화 설정
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 11

## Step 2: DINO 모델 로드 🤖

In [ ]:
print("\n" + "="*60)
print("🤖 DINO 모델 다운로드 중...")
print("="*60)

print("\n📥 DINO (ViT-S/16) 로딩...")
print("   - Self-supervised Vision Transformer")
print("   - 논문의 '예외' 케이스: Clean attention maps!")

# DINO 모델 로드
dino = torch.hub.load('facebookresearch/dino:main', 'dino_vits16')
dino = dino.to(device).eval()

# 모델 정보
num_params = sum(p.numel() for p in dino.parameters())
print(f"\n✅ 로드 완료!")
print(f"   Parameters: {num_params:,}")
print(f"   Patch size: 16x16")
print(f"   Model: ViT-Small")

print("\n" + "="*60)
print("🎉 준비 완료!")
print("="*60)

## Step 3: 이미지 준비 🖼️

### 옵션 A: 샘플 이미지 사용 (기본)

In [ ]:
# 샘플 이미지 다운로드
print("📥 샘플 이미지 다운로드...")
url = 'https://raw.githubusercontent.com/facebookresearch/dinov2/main/assets/example.jpg'
urllib.request.urlretrieve(url, 'input.jpg')
print("✅ 완료!")

img = Image.open('input.jpg').convert('RGB')
print(f"이미지 크기: {img.size}")

### 옵션 B: 자신의 이미지 업로드 (선택)

In [ ]:
# 이 셀 실행 시 파일 업로드 가능
# uploaded = files.upload()
# img = Image.open(list(uploaded.keys())[0]).convert('RGB')
# print(f"✅ 업로드 완료: {img.size}")

In [ ]:
# 이미지 전처리
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

img_tensor = transform(img).unsqueeze(0).to(device)

# 원본 이미지 표시
plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.title('Input Image', fontsize=16, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.savefig('input_image.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ 이미지 준비 완료!")

## Step 4: Attention Maps 추출 🔍

In [ ]:
print("\n" + "="*60)
print("🔍 Attention Maps 추출 중...")
print("="*60 + "\n")

# Attention 추출
with torch.no_grad():
    attentions = dino.get_last_selfattention(img_tensor)

print(f"📊 Raw attention shape: {attentions.shape}")
print(f"   [batch, heads, tokens, tokens]")

# [CLS] token에서 patch tokens로의 attention
nh = attentions.shape[1]  # number of heads
attentions = attentions[0, :, 0, 1:].reshape(nh, -1)

# Reshape to spatial grid
patch_size = 16
w_featmap = img_tensor.shape[-2] // patch_size
h_featmap = img_tensor.shape[-1] // patch_size

attentions = attentions.reshape(nh, w_featmap, h_featmap)

# Interpolate to image size
attentions = nn.functional.interpolate(
    attentions.unsqueeze(0),
    scale_factor=patch_size,
    mode='nearest'
)[0].cpu().numpy()

print(f"\n✅ 처리 완료!")
print(f"   Final shape: {attentions.shape}")
print(f"   Heads: {nh}")
print(f"   Spatial: {attentions.shape[1]}x{attentions.shape[2]}")

# Artifacts 분석
mean_val = attentions.mean()
std_val = attentions.std()
high_norm_ratio = (attentions > mean_val + 2*std_val).mean() * 100

print(f"\n📈 통계:")
print(f"   Mean: {mean_val:.6f}")
print(f"   Std: {std_val:.6f}")
print(f"   Range: [{attentions.min():.6f}, {attentions.max():.6f}]")
print(f"   High-norm tokens: {high_norm_ratio:.2f}%")

if high_norm_ratio < 1.0:
    print(f"\n✅ CLEAN! Artifacts 거의 없음 (< 1%)")
else:
    print(f"\n⚠️  Some artifacts detected ({high_norm_ratio:.2f}%)")

print("\n" + "="*60)

## Step 5: Attention Maps 시각화 🎨

In [ ]:
print("\n🎨 시각화 생성 중...\n")

# 6개 head 시각화
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i in range(6):
    # Attention map
    im = axes[i].imshow(attentions[i], cmap='viridis')
    axes[i].set_title(f'Head {i+1}', fontsize=14, fontweight='bold')
    axes[i].axis('off')
    
    # Colorbar
    plt.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)

plt.suptitle(
    'DINO Attention Maps (ViT-S/16)\n' +
    f'Clean attention without artifacts (High-norm: {high_norm_ratio:.2f}%)',
    fontsize=16, fontweight='bold', y=0.98
)

plt.tight_layout(rect=[0, 0.01, 1, 0.96])
plt.savefig('dino_attention_maps.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ 시각화 완료!")

## Step 6: 종합 시각화 📊

In [ ]:
fig = plt.figure(figsize=(20, 10))
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

# 1. Original image
ax1 = fig.add_subplot(gs[0, 0])
ax1.imshow(img)
ax1.set_title('Input Image', fontsize=14, fontweight='bold')
ax1.axis('off')

# 2. Average attention
ax2 = fig.add_subplot(gs[0, 1])
avg_att = attentions.mean(axis=0)
im = ax2.imshow(avg_att, cmap='viridis')
ax2.set_title('Average Attention\n(All heads)', fontsize=14, fontweight='bold')
ax2.axis('off')
plt.colorbar(im, ax=ax2, fraction=0.046)

# 3. Max attention
ax3 = fig.add_subplot(gs[0, 2])
max_att = attentions.max(axis=0)
im = ax3.imshow(max_att, cmap='hot')
ax3.set_title('Max Attention\n(Peak values)', fontsize=14, fontweight='bold')
ax3.axis('off')
plt.colorbar(im, ax=ax3, fraction=0.046)

# 4. Attention histogram
ax4 = fig.add_subplot(gs[1, 0])
ax4.hist(attentions.flatten(), bins=100, alpha=0.7, color='blue', edgecolor='black')
ax4.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.4f}')
ax4.axvline(mean_val + 2*std_val, color='orange', linestyle='--', linewidth=2, 
           label=f'Mean+2σ: {mean_val + 2*std_val:.4f}')
ax4.set_xlabel('Attention Value', fontsize=12, fontweight='bold')
ax4.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax4.set_title('Attention Distribution', fontsize=14, fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

# 5. Statistics table
ax5 = fig.add_subplot(gs[1, 1:])
ax5.axis('off')

stats_text = f"""
DINO (ViT-S/16) 분석 결과
{'='*50}

📊 Attention 통계:
   Mean:      {mean_val:.6f}
   Std:       {std_val:.6f}
   Min:       {attentions.min():.6f}
   Max:       {attentions.max():.6f}

🎯 Artifacts 분석:
   High-norm ratio: {high_norm_ratio:.2f}%
   Status: {'✅ CLEAN (< 1%)' if high_norm_ratio < 1.0 else '⚠️  Some artifacts'}

🔬 모델 정보:
   Model: DINO (Self-supervised ViT)
   Architecture: ViT-Small/16
   Patch size: 16x16
   Heads: {nh}
   Parameters: {num_params:,}

📚 논문:
   "Emerging Properties in Self-Supervised
    Vision Transformers" (ICCV 2021)

💡 특징:
   - 예외적으로 clean attention maps
   - 의미있는 semantic segmentation
   - 높은 feature 품질
"""

ax5.text(0.1, 0.5, stats_text, 
        fontsize=11, verticalalignment='center',
        fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))

plt.suptitle('DINO Complete Analysis Dashboard', 
            fontsize=18, fontweight='bold', y=0.98)

plt.savefig('dino_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ 대시보드 생성 완료!")

## Step 7: 개별 Head 분석 🔬

In [ ]:
print("\n" + "="*60)
print("🔬 Individual Head Analysis")
print("="*60 + "\n")

for i in range(nh):
    head_att = attentions[i]
    head_mean = head_att.mean()
    head_std = head_att.std()
    head_artifacts = (head_att > head_mean + 2*head_std).mean() * 100
    
    status = "✅ Clean" if head_artifacts < 1.0 else "⚠️  Some artifacts"
    
    print(f"Head {i+1:2d}: "
          f"Mean={head_mean:.6f}, "
          f"Std={head_std:.6f}, "
          f"Artifacts={head_artifacts:5.2f}% {status}")

print("\n" + "="*60)

## Step 8: 결과 다운로드 💾

In [ ]:
# 생성된 파일 확인
output_files = [
    'input_image.png',
    'dino_attention_maps.png',
    'dino_dashboard.png'
]

print("\n📁 생성된 파일:")
for f in output_files:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024
        print(f"  ✅ {f} ({size:.1f} KB)")

print("\n💾 다운로드 방법:")
print("  1. 왼쪽 폴더 아이콘 클릭")
print("  2. 파일 우클릭 → 다운로드")
print("\n또는 아래 셀 실행:")

In [ ]:
# ZIP으로 묶어서 다운로드
!zip -q dino_results.zip input_image.png dino_attention_maps.png dino_dashboard.png

print("✅ dino_results.zip 생성 완료!")
print("\n다운로드 중...")
files.download('dino_results.zip')

## 🎉 완료!

### 📊 실험 요약

#### DINO의 특징
- ✅ **Clean attention maps**: Artifacts < 1%
- ✅ **의미있는 segmentation**: 객체 경계 명확
- ✅ **높은 feature 품질**: Self-supervised 학습

#### 왜 DINO가 특별한가?

"Vision Transformers Need Registers" 논문에서:
> "All models BUT DINO exhibit peaky outlier values"

DINO만 예외적으로 artifacts 없음!

가능한 이유:
1. **Model size**: ViT-Small (작은 모델)
2. **Training data**: ImageNet-1K
3. **Self-distillation**: 특별한 학습 방식
4. **Training length**: 짧은 학습 기간

### 📚 관련 논문

#### 핵심 논문
1. **DINO** (ICCV 2021)
   - "Emerging Properties in Self-Supervised Vision Transformers"
   - https://arxiv.org/abs/2104.14294

2. **Register Tokens** (ICLR 2024)
   - "Vision Transformers Need Registers"
   - https://arxiv.org/abs/2309.16588

#### 후속 연구
3. **DINOv2** (2023)
   - Larger scale, but artifacts appear
   - https://arxiv.org/abs/2304.07193

### 💡 활용 방법

```python
# DINO features 추출
with torch.no_grad():
    features = dino(img_tensor)

# Attention maps 추출
attention = dino.get_last_selfattention(img_tensor)

# Object discovery (LOST 등)
# Semantic segmentation
# Feature matching
```

### 🔬 확장 실험

- 다양한 이미지로 테스트
- 다른 모델 크기 비교 (ViT-S, B, L)
- DINOv2와 비교 분석
- Register tokens 효과 검증

### 📖 인용

```bibtex
@inproceedings{caron2021emerging,
  title={Emerging Properties in Self-Supervised Vision Transformers},
  author={Caron, Mathilde and Touvron, Hugo and Misra, Ishan and 
          J{\'e}gou, Herv{\'e} and Mairal, Julien and Bojanowski, Piotr and Joulin, Armand},
  booktitle={ICCV},
  year={2021}
}
```

---

**문제 발생 시:**
- GPU 없음 → 런타임 유형 변경
- 메모리 부족 → 세션 다시 시작
- 모델 로드 실패 → 인터넷 연결 확인